<a href="https://colab.research.google.com/github/rongione/anchorage_prediction/blob/main/anchorage_prediction_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statistics import mean, stdev
from google.colab import drive

In [ ]:
drive.mount('/content/drive')

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/boats_at_harbor/boats_at_harbor.csv")

# Data preparation

In [ ]:
df['datum'] = pd.to_datetime(df['datum'], utc=True)

df["windDirectionDegrees"] = pd.to_numeric(df["windDirectionDegrees"], errors="coerce")
df["windSpeedMps"] = pd.to_numeric(df["windSpeedMps"], errors="coerce")
df["temperatureCelsius"] = pd.to_numeric(df["temperatureCelsius"], errors="coerce")
df["boats_at_harbor"] = pd.to_numeric(df["boats_at_harbor"], errors="coerce")
df["day_of_week"] = pd.to_numeric(df["day_of_week"], errors="coerce")
df["precipitationMm"] = pd.to_numeric(df["precipitationMm"], errors="coerce")

# Feature engineering

### Hour cos and sin

In [ ]:
df['hour'] = df['datum'].dt.hour
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

### Wind cos and sin

In [ ]:
wind_rad = np.deg2rad(df['windDirectionDegrees'])
df["wind_dir_sin"] = np.sin(wind_rad)
df["wind_dir_cos"] = np.cos(wind_rad)

In [ ]:
# Verify rows
df.info()
df.head(40)

### Hamnar med flest båtar

Följande kod kommer att gruppera data per hamn och räkna antalet båtar i varje hamn, och sedan visa de 100 hamnar med flest båtar.

In [ ]:
top_harbors = df.groupby('hamnId')['boats_at_harbor'].sum().nlargest(100)
display(top_harbors)
print(f"Antal hamnar i top_harbors: {len(top_harbors)}")

# Korrelation mellan alla relevanta variabler och antal båtar

In [ ]:
top_harbor_ids = top_harbors.index

# Filtrera DataFrame för att endast inkludera data från de 100 största hamnarna
df_top_harbors = df[df['hamnId'].isin(top_harbor_ids)]

# Välj ut relevanta numeriska kolumner för korrelationsanalys
relevant_cols = [
    'boats_at_harbor',
    #'average_boats_at_harbor', # Borttagen för att förhindra dataläckage
    'temperatureCelsius',
    'day_of_week',
    'hour_sin',
    'hour_cos',
    'wind_dir_sin',
    'wind_dir_cos',
    #'boats_at_harbor_lag'
]

# Beräkna korrelationsmatrisen
correlation_matrix = df_top_harbors[relevant_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap='coolwarm',
    fmt=".2f",
    linewidths=.5
)
plt.title('Korrelationsmatris för relevanta variabler och antal båtar')
plt.show()

# Beskrivning av hela datasetet

In [ ]:
# Shape of dataset
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

# Data types
print("\nData types:")
print(df.dtypes)

# Missing values
print("\nMissing values per column:")
print(df.isnull().sum())

# Duplicate rows
print(f"\nDuplicate rows: {df.duplicated().sum()}")

# Unique values in categorical columns
print("\nUnique values:")
for col in ["hamnId", "weatherType", "day_of_week"]:
    print(f"{col}: {df[col].nunique()} unique values")

# Check for invalid ranges
print("\nRange validation:")
print("Negative wind speeds:", (df["windSpeedMps"] < 0).sum())
print("Invalid wind directions:", ((df["windDirectionDegrees"] < 0) |
                                   (df["windDirectionDegrees"] > 360)).sum())
print("Negative precipitation:", (df["precipitationMm"] < 0).sum())

# Distribution of target variable
print("\nBoat count distribution:")
print(df["boats_at_harbor"].value_counts().sort_index())

# Beskrivning av datasetet (Top 100 hamnar)

In [ ]:
# Shape of dataset
print(f"Rows: {df_top_harbors.shape[0]}, Columns: {df_top_harbors.shape[1]}")

# Data types
print("\nData types:")
print(df_top_harbors.dtypes)

# Missing values
print("\nMissing values per column:")
print(df_top_harbors.isnull().sum())

# Duplicate rows
print(f"\nDuplicate rows: {df_top_harbors.duplicated().sum()}")

# Summary statistics for numerical columns
print("\nNumerical summary:")
print(df_top_harbors.describe())

# Unique values in categorical columns
print("\nUnique values:")
for col in ["hamnId", "weatherType", "day_of_week"]:
    print(f"{col}: {df_top_harbors[col].nunique()} unique values")

# Check for invalid ranges
print("\nRange validation:")
print("Negative wind speeds:", (df_top_harbors["windSpeedMps"] < 0).sum())
print("Invalid wind directions:", ((df_top_harbors["windDirectionDegrees"] < 0) |
                                   (df_top_harbors["windDirectionDegrees"] > 360)).sum())
print("Negative precipitation:", (df_top_harbors["precipitationMm"] < 0).sum())

# Distribution of target variable
print("\nBoat count distribution:")
print(df_top_harbors["boats_at_harbor"].value_counts().sort_index())

# Konstruktion av attributet genomsnittlig beläggning (enbart på träningsdatat)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

# Välj ut relevanta funktioner och målvariabel
# Inkludera 'hamnId' och 'boats_at_harbor' temporärt i X för beräkning av medelantalet båtar, för att förhindra dataläckage
# boats_at_harbor kommer att tas bort från X_train och X_test efter beräkning av average_boats_at_harbor
features_for_split = [
    'hamnId',
    'boats_at_harbor', # Temporarily included for calculating average_boats_at_harbor
    'windSpeedMps',
    'temperatureCelsius',
    'day_of_week',
    'hour_sin',
    'hour_cos',
    'wind_dir_sin',
    'wind_dir_cos'
]
target = 'boats_at_harbor'

X = df_top_harbors[features_for_split] # Use the new list with boats_at_harbor
y = df_top_harbors[target]

# Dela upp data i tränings- och testset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Träningsset storlek: {len(X_train)} rader")
print(f"Testset storlek: {len(X_test)} rader")

In [ ]:
# Beräkna 'average_boats_at_harbor' endast från träningsdata för att förhindra dataläckage
# Använd 'target' variabeln som referens för 'boats_at_harbor'
avg_boats_per_harbor = X_train.groupby('hamnId')[target].mean()

# Mappa dessa medelvärden till både tränings- och testsetet
X_train['average_boats_at_harbor'] = X_train['hamnId'].map(avg_boats_per_harbor)
X_test['average_boats_at_harbor'] = X_test['hamnId'].map(avg_boats_per_harbor)

# Hantera hamnar i testsetet som inte finns i träningssetet (sätt till 0)
X_test['average_boats_at_harbor'].fillna(0, inplace=True)

# Ta bort den temporära 'hamnId' kolumnen och den ursprungliga 'boats_at_harbor' från X_train och X_test
X_train = X_train.drop(columns=['hamnId', target]) # Drop 'target' (boats_at_harbor) from X_train
X_test = X_test.drop(columns=['hamnId', target])   # Drop 'target' (boats_at_harbor) from X_test

# Uppdatera till de slutgiltiga features för modellträningen
features = [
    'windSpeedMps',
    'temperatureCelsius',
    'day_of_week',
    'hour_sin',
    'hour_cos',
    'wind_dir_sin',
    'wind_dir_cos',
    'average_boats_at_harbor'
]

print("average_boats_at_harbor har beräknats och lagts till. hamnId och boats_at_harbor har tagits bort från X.")

# Standardisering

In [ ]:
# Initiera StandardScaler
scaler = StandardScaler()

# Anpassa scalern till träningsdata och transformera både tränings- och testdata
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Konvertera tillbaka till DataFrame för att behålla kolumnnamn
X_train_scaled = pd.DataFrame(X_train_scaled, columns=features, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=features, index=X_test.index)

print("Data har standardiserats.")

### Träna den linjära regressionsmodellen

In [ ]:
# Initiera och träna modellen
model = LinearRegression()
model.fit(X_train_scaled, y_train)

print("Modellen har tränats klart: LinearRegression.")

### Träna RandomForest modellen

In [ ]:
# Initiera och träna modellen
regressor = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    oob_score=True
)
regressor.fit(X_train_scaled, y_train)

print("Modellen har tränats klart: RandomForest.")

# Utvärdera RandomForest

In [ ]:
# Out-of-Bag Score: Measures how well the model
# generalizes on unseen samples, a low value indicates weaker generalization.
print("Out-of-Bag Score:", regressor.oob_score_)

y_pred_rf = regressor.predict(X_test_scaled)

mae_rf = mean_absolute_error(y_test, y_pred_rf)
mse_rf = mean_squared_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mse_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print(f"Mean Absolute Error (MAE) (RandomForest): {mae_rf:.2f}")
print(f"Mean Squared Error (MSE) (RandomForest): {mse_rf:.2f}")
print(f"Root Mean Squared Error (RMSE) (RandomForest): {rmse_rf:.2f}")
print(f"R-squared (R²) (RandomForest): {r2_rf:.2f}")

# Utvärdera LinearRegression

In [ ]:
y_pred_lr = model.predict(X_test_scaled)

# Sätt negativa prediktioner till noll, eftersom antal båtar inte kan vara negativt
y_pred_lr[y_pred_lr < 0] = 0

mae_lr = mean_absolute_error(y_test, y_pred_lr)
mse_lr = mean_squared_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mse_lr)
r2_lr = r2_score(y_test, y_pred_lr)

print(f"Mean Absolute Error (MAE) (LinearRegression): {mae_lr:.2f}")
print(f"Mean Squared Error (MSE) (LinearRegression): {mse_lr:.2f}")
print(f"Root Mean Squared Error (RMSE) (LinearRegression): {rmse_lr:.2f}")
print(f"R-squared (R²) (LinearRegression): {r2_lr:.2f}")

In [ ]:
results = pd.DataFrame({'Actual': y_test, 'Predicted (LinearRegression)': y_pred_lr, 'Predicted (RandomForest)': y_pred_rf})
display(results.head(40))

# Scatter plot (Linjär Regression)

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(x=results['Actual'], y=results['Predicted (LinearRegression)'], alpha=0.6)
plt.plot([y.min(), y.max()], [y.min(), y.max()], color='green', linestyle='--') # Perfekt förutsägelselinje
plt.xlabel('Faktiska värden')
plt.title('Predikterade vs Faktiska värden (Linjär regression)', fontsize=14)
plt.grid(True, linestyle='--', alpha=0.5)
plt.ylabel('Predikterade värden')
plt.show()

# Distribution av Residualer (Linjär Regression)

In [ ]:
residuals_lr = y_test - y_pred_lr

plt.figure(figsize=(10, 6))
plt.hist(residuals_lr, bins=50)
sns.histplot(residuals_lr, kde=True, bins=50)
plt.title('Distribution av Residualer (Linjär regression)')
plt.xlabel('Residualer')
plt.grid(True, linestyle='--', alpha=0.5)
plt.ylabel('Frekvens')
plt.show()

# Scatter plot (RandomForest)

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(x=results['Actual'], y=results['Predicted (RandomForest)'], alpha=0.6)
plt.plot([y.min(), y.max()], [y.min(), y.max()], color='green', linestyle='--') # Perfekt förutsägelselinje
plt.title('Faktiska vs. Predikterade värden (Random Forest)')
plt.xlabel('Faktiska värden')
plt.ylabel('Predikterade värden')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

# Distribution av Residualer (RandomForest)




In [ ]:
residuals_rf = y_test - y_pred_rf

plt.figure(figsize=(10, 6))
plt.hist(residuals_rf, bins=50)
sns.histplot(residuals_rf, kde=True, bins=50)
plt.title('Distribution av Residualer (RandomForest)')
plt.xlabel('Residualer')
plt.grid(True, linestyle='--', alpha=0.5)
plt.ylabel('Frekvens')
plt.show()

### Fördelning av målvariabeln (hela datasetet)

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df['boats_at_harbor'], kde=True, bins=50)
plt.title('Fördelning av Boats at Harbor (Hela)')
plt.xlabel('Antal Båtar i Hamn')
plt.ylabel('Frekvens')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

### Fördelning av målvariabeln (Top 100 Hamnar)

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df_top_harbors['boats_at_harbor'], kde=True, bins=50)
plt.title('Fördelning av Boats at Harbor (Top 100 Hamnar)')
plt.xlabel('Antal Båtar i Hamn')
plt.ylabel('Frekvens')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

# Toppfunktioner från RandomForest-modellen

Detta diagram visar de viktigaste funktionerna som används av RandomForest-modellen för att göra förutsägelser.

In [ ]:
feature_importances = regressor.feature_importances_
features_df = pd.DataFrame({'Feature': features, 'Importance': feature_importances})
features_df = features_df.sort_values(by='Importance', ascending=False)

plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', data=features_df)
plt.title('Figure 5.3: Feature Importances from RandomForestRegressor')
plt.xlabel('Betydelse')
plt.ylabel('Funktion')
plt.show()